# 🧠 Why LLMs Forget: Conversation State Is a Message List

## Learning Objectives
In this notebook, you will learn:
1. **Statelessness** - see a chat model forget your name between two calls
2. **Manual history** - make it "remember" by resending the earlier messages yourself
3. **Chat message histories** - keep those messages in an `InMemoryChatMessageHistory` object instead of a loose list
4. **Message types** - tell `SystemMessage`, `HumanMessage` and `AIMessage` apart and filter a history by type

## Where this fits
This is the first notebook of the module. Every memory technique that follows — classic memory classes,
`RunnableWithMessageHistory`, SQLite persistence, trimming and summaries — is a different way of deciding
*which* earlier messages to send back to the model. This notebook shows why that is necessary at all.

## Prerequisites
- LangChain basics: prompts, chat models and the `|` (LCEL) operator
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

> **⚠️ Legacy API**: Part 3's `InMemoryChatMessageHistory` is deprecated since `langchain-core` 1.6.4 and removed in 2.0. LangChain now recommends LangGraph persistence instead — a checkpointer
> and a `thread_id` — taught in [`01_memory_and_conversational_agent.ipynb`](06a_memory_and_conversational_agent.ipynb)
> (module 02). This module keeps the older API because existing LCEL code uses it everywhere; the setup cell
> lets its deprecation warnings through so you see them where the deprecated classes are used.


### What "memory" means in LangChain
Memory is the ability of an application to store and retrieve information about previous interactions:
- **Context retention**: details of earlier questions and answers keep the conversation coherent
- **Flexible storage**: the history can live in a Python list, an in-memory store, a database or a file
- **Scalability**: the same interface works for one user in a notebook and for many users in production

Without memory, every query is treated in isolation; with it, the application can recall preferences,
earlier questions and any context shared during the conversation.

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library warnings — except LangChain's deprecation warnings, which it lets
through because this notebook uses deprecated APIs on purpose.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv
from langchain_core._api import LangChainDeprecationWarning

warnings.filterwarnings("ignore")
# This notebook uses deprecated LangChain APIs on purpose: let their deprecation warnings through
warnings.filterwarnings("default", category=LangChainDeprecationWarning)

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the chat model used by every example
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧊 Part 1: A Chat Model Is Stateless

A chat model call is a pure function of the messages you send in that call. Nothing is stored between calls,
so a second call cannot know what you said in the first one. The chain below is `prompt | llm | parser` —
no memory component anywhere.

In [ ]:
# ============================================================================
# PART 1: TWO CALLS, NO MEMORY
# ============================================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

stateless_chain = ChatPromptTemplate.from_template("{prompt}") | llm | StrOutputParser()

# Turn 1: introduce yourself
print("User: My name is Sourav and I work at Databricks.")
print("AI:  ", stateless_chain.invoke({"prompt": "My name is Sourav and I work at Databricks."}))

# Turn 2: ask the model to recall it — a completely independent call
print("\nUser: What's my name?")
print("AI:  ", stateless_chain.invoke({"prompt": "What's my name?"}))

The second answer cannot contain your name: the model only received the text "What's my name?".
Anything that looks like memory in a chatbot is the application resending earlier messages.

---
## ✍️ Part 2: Remembering by Resending Messages

Chat models accept a **list of messages**. If that list contains the earlier turns, the model can use them.
This is all memory really is — the rest of the module is about managing that list well.

In [ ]:
# ============================================================================
# PART 2A: PASS THE EARLIER TURNS EXPLICITLY
# ============================================================================
from langchain_core.messages import AIMessage, HumanMessage

response = llm.invoke([
    HumanMessage(content="Hi, my name is Sourav and I am a Resident Solution Architect at Databricks."),
    AIMessage(content="Hello Sourav! Nice to meet you."),
    HumanMessage(content="What's my name and what do I do?"),
])
print(response.content)

The model now answers correctly because the earlier turns are part of *this* call. Doing that by hand gets
tedious, so the next cell wraps it in a tiny helper that appends every turn to a Python list.

In [ ]:
# ============================================================================
# PART 2B: A HAND-ROLLED MEMORY LOOP
# ============================================================================
history = []  # the entire "memory" of this chatbot


def chat(user_input: str) -> str:
    """Send the whole history plus the new message, then store both sides of the turn."""
    history.append(HumanMessage(content=user_input))
    reply = llm.invoke(history)
    history.append(reply)  # reply is an AIMessage
    return reply.content


for turn in ["Hi, I'm Priya and I love hiking.", "I'm planning a trip to Norway.", "What do you know about me?"]:
    print(f"User: {turn}")
    print(f"AI:   {chat(turn)}\n")

print(f"Messages sent on the last call: {len(history) - 1}")

Notice the cost: every call resends everything said so far, so token usage grows with the conversation.
Bounding that growth is the subject of `05_trimming_windowing_and_summary.ipynb`.

---
## 🗂️ Part 3: Chat Message Histories

LangChain gives the list a proper home: a **chat message history** object with `add_user_message`,
`add_ai_message`, `messages` and `clear`. `InMemoryChatMessageHistory` keeps messages in RAM; later notebooks
swap in SQLite- and file-backed histories with the same interface.

### Key Concepts:
- **`BaseChatMessageHistory`**: the interface every history backend implements
- **`InMemoryChatMessageHistory`**: the in-process implementation (`ChatMessageHistory` in `langchain_community` is an alias you will see in older code)

In [ ]:
# ============================================================================
# PART 3: A CHAT MESSAGE HISTORY OBJECT
# ============================================================================
from langchain_core.chat_history import InMemoryChatMessageHistory

store = InMemoryChatMessageHistory()
store.add_user_message("hi!")
store.add_ai_message("hello my friend!")

for message in store.messages:
    print(f"{message.type:>5}: {message.content}")

# The same helper as Part 2, now backed by the history object
def chat_with_store(user_input: str) -> str:
    store.add_user_message(user_input)
    reply = llm.invoke(store.messages)
    store.add_ai_message(reply.content)
    return reply.content


print("\nUser: I just said hi. What did you answer?")
print("AI:  ", chat_with_store("I just said hi. What did you answer?"))

---
## 🏷️ Part 4: Message Types

Every entry in a history is a typed message. The type tells the model who said it:
- **`SystemMessage`**: instructions for the model; usually not part of the stored conversation
- **`HumanMessage`**: what the user said
- **`AIMessage`**: what the model answered

In [ ]:
# ============================================================================
# PART 4: FILTER A HISTORY BY MESSAGE TYPE
# ============================================================================
from langchain_core.messages import SystemMessage

conversation = [SystemMessage(content="You are a concise assistant.")] + history

ai_messages = [m for m in conversation if isinstance(m, AIMessage)]
human_messages = [m for m in conversation if isinstance(m, HumanMessage)]

print(f"System: {conversation[0].content}")
print(f"\nHuman messages ({len(human_messages)}):")
for m in human_messages:
    print(f"  - {m.content}")
print(f"\nAI messages ({len(ai_messages)}):")
for m in ai_messages:
    print(f"  - {m.content[:80]}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Statelessness
- A chat model call sees only the messages sent in that call; it keeps nothing between calls

### 2. Manual history
- Resending earlier `HumanMessage`/`AIMessage` pairs is what makes a model "remember"
- Every turn resends the whole history, so cost grows with the conversation

### 3. Chat message histories
- `InMemoryChatMessageHistory` stores the list behind a standard interface that other backends share

### 4. Message types
- System, human and AI messages carry roles; histories can be filtered by type

### Next Steps
- **`02_classic_memory_classes_and_chains.ipynb`**: LangChain's classic memory classes, on their own and inside chains